# Feature Blocks Load Debug

Отдельный ноутбук только для поблочной загрузки фичей.

Ожидается, что перед запуском этих ячеек уже существуют переменные:

- `df` — датасет с `contact_id` и таргетом;
- `aud_query` — SQL аудитории или `select * from temp_aud_table`;
- `engine`;
- `session`;
- `state`;
- `event_timestamp`;
- `feature_date`;
- `preperiod_months`;
- `fav_omni_features_table`;
- `uplift_rate_prefix`;
- `aud_suffix`.

Идея: запускать ячейки по одной и смотреть, сколько времени грузится каждый блок, какие колонки добавились и не размножились ли строки.

## Imports And Helper

In [ ]:
import time

import numpy as np
import pandas as pd
import pyspark.sql.functions as F

import magpie.sql_utils as su

import cvm_model.utils as utils
import cvm_model.sql as sql

pd.set_option('display.max_columns', None)


In [ ]:
def merge_block(df, df_part, block_name):
    before_rows, before_cols = df.shape
    part_rows, part_cols = df_part.shape
    new_cols = [c for c in df_part.columns if c != 'contact_id' and c not in df.columns]

    t0 = time.time()
    df = df.merge(df_part, on='contact_id', how='left')
    merge_sec = time.time() - t0

    print(
        f'{block_name}: part={part_rows:,}x{part_cols:,}; '
        f'df={before_rows:,}x{before_cols:,} -> {df.shape[0]:,}x{df.shape[1]:,}; '
        f'added={len(new_cols)}; merge_sec={merge_sec:.1f}'
    )
    print('new_cols:', new_cols)

    assert df.shape[0] == before_rows, f'{block_name}: merge changed row count!'
    assert df['contact_id'].nunique() == len(df), f'{block_name}: contact_id duplicates after merge!'

    return df


def run_sql_block(query, block_name, stream=False):
    print(f'Loading {block_name}...')
    t0 = time.time()
    if stream:
        df_part = utils.get_df_stream(engine, query)
    else:
        df_part = utils.get_df(engine, query)
    sec = time.time() - t0
    print(f'{block_name}: SQL loaded in {sec:.1f} sec; shape={df_part.shape}')
    return df_part


## 1. Recency

In [ ]:
query_kwargs = dict(
    aud=aud_query,
    date=feature_date,
    month=preperiod_months[0],
)

df_part = run_sql_block(
    sql.recency_query.format(**query_kwargs),
    block_name='recency',
)

df = merge_block(df, df_part, 'recency')
display(df_part.head())


## 2. Favourite OMNI Temp Table

Этот шаг нужен до блока `fav_omni_features`.

In [ ]:
query_kwargs = dict(
    aud=aud_query,
    date=feature_date,
    month=preperiod_months[0],
)

query = sql.fav_omni_features_create_query.format(**query_kwargs)
create_query = sql.create_table_from_select_query.format(
    table=fav_omni_features_table,
    query=query,
    distribution_col='contact_id',
)

t0 = time.time()
utils.execute_query(engine, f'drop table if exists {fav_omni_features_table}')
utils.execute_query(engine, create_query)
print(f'Favourite OMNI temp table created in {time.time() - t0:.1f} sec')


## 3. Cheque Long / Store Context

In [ ]:
month = preperiod_months[0]
df_part = run_sql_block(
    sql.cheque_query.format(aud=aud_query, date=feature_date, month=month),
    block_name=f'cheque_long_{month}m',
    stream=True,
)

df = merge_block(df, df_part, f'cheque_long_{month}m')
display(df_part.head())


## 4. Cheque Short Windows

In [ ]:
for month in preperiod_months[1:]:
    df_part = run_sql_block(
        sql.cheque_query_short.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'cheque_short_{month}m',
        stream=True,
    )
    df = merge_block(df, df_part, f'cheque_short_{month}m')

display(df.head())


## 5. App Logins

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.app_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'app_{month}m',
        stream=True,
    )
    df = merge_block(df, df_part, f'app_{month}m')

display(df.head())


## 6. OMNI QR

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.omni_qr_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'omni_qr_{month}m',
    )
    df = merge_block(df, df_part, f'omni_qr_{month}m')

display(df.head())


## 7. OMNI Features

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.omni_features_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'omni_features_{month}m',
    )
    df = merge_block(df, df_part, f'omni_features_{month}m')

display(df.head())


## 8. Favourite OMNI Features

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.fav_omni_features_select_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
            fav_omni_features_table=fav_omni_features_table,
        ),
        block_name=f'fav_omni_{month}m',
    )
    df = merge_block(df, df_part, f'fav_omni_{month}m')

display(df.head())


## 9. Unique OMNI Features

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.omni_unique_features_count_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'omni_unique_{month}m',
    )
    df = merge_block(df, df_part, f'omni_unique_{month}m')

display(df.head())


## 10. OMNI Goals / Missions

В `load_features` этот блок грузится для `preperiod_months[1:]`.

In [ ]:
for month in preperiod_months[1:]:
    df_part = run_sql_block(
        sql.omni_goals_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'omni_goals_{month}m',
        stream=True,
    )
    df = merge_block(df, df_part, f'omni_goals_{month}m')

display(df.head())


## 11. Accepts

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.accept_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'accepts_{month}m',
        stream=True,
    )
    df = merge_block(df, df_part, f'accepts_{month}m')

display(df.head())


## 12. Bonuses

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.bonus_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'bonuses_{month}m',
    )
    df = merge_block(df, df_part, f'bonuses_{month}m')

display(df.head())


## 13. Loyalty Level

In [ ]:
for month in preperiod_months:
    df_part = run_sql_block(
        sql.level_query.format(aud=aud_query, date=feature_date, month=month),
        block_name=f'level_{month}m',
    )
    df = merge_block(df, df_part, f'level_{month}m')

display(df.head())


## 14. Tendency Features

In [ ]:
calc_period = preperiod_months[0]

df['transaction_tendency'] = df['cheque_recency'] / np.max((np.ones(len(df)), df[f'trans_lag_avg_{calc_period}']), axis=0)
df['login_tendency'] = df['login_recency'] / np.max((np.ones(len(df)), df[f'login_lag_avg_{calc_period}']), axis=0)
df['omni_qr_tendency'] = df['omni_qr_recency'] / np.max((np.ones(len(df)), df[f'omni_qr_lag_avg_{calc_period}']), axis=0)
df['omni_features_tendency'] = df['omni_features_recency'] / np.max((np.ones(len(df)), df[f'omni_features_lag_avg_{calc_period}']), axis=0)

print('After tendency:', df.shape)
display(df[['transaction_tendency', 'login_tendency', 'omni_qr_tendency', 'omni_features_tendency']].head())


## 15. Static Client Features

In [ ]:
df_part = run_sql_block(
    sql.static_features_query.format(aud=aud_query, date=feature_date, month=preperiod_months[0]),
    block_name='static_client',
)

df = merge_block(df, df_part, 'static_client')
display(df_part.head())


## 16. DAC History / DAC Segments

In [ ]:
df_part = run_sql_block(
    sql.dac_months_count_query.format(aud=aud_query, date=feature_date, month=preperiod_months[0]),
    block_name='dac_history_segments',
)

df = merge_block(df, df_part, 'dac_history_segments')

df.loc[df['dac_age_months'] == 0, 'dac_age_months'] = 1
df['dac_months_per_dac_age_ratio'] = df['dac_months_count'] / df['dac_age_months']

print('After DAC history ratio:', df.shape)
display(df_part.head())


## 17. Uplift Rate

Этот блок повторяет кусок из `utils.load_features`. Он может быть тяжёлым и зависит от S3/Spark.

In [ ]:
aud_prefix = state.settings.preprocess_prefix(event_timestamp) / aud_suffix

uplift_rate_bucket = uplift_rate_prefix.split('//')[1].split('/')[0]
uplift_rate_key = '/'.join(uplift_rate_prefix.split('//')[1].split('/')[1:])

aud_bucket = aud_prefix.split('//')[1].split('/')[0]
aud_key = '/'.join(aud_prefix.split('//')[1].split('/')[1:])

uplift_rate_table = f's3a://{uplift_rate_bucket}/{uplift_rate_key}'
aud_s3_table = f's3a://{aud_bucket}/{aud_key}'

t0 = time.time()
su.save_to_s3(
    input=aud_query,
    prefix=aud_key,
    input_type='query',
    delete=True,
    broadcast=True,
    bucket=aud_bucket,
)

aud = session.read.parquet(aud_s3_table)
uplift_rate = session.read.parquet(uplift_rate_table).filter(F.col('finish_date') < feature_date)

df_part = aud.join(uplift_rate, 'contact_id', 'inner').toPandas().fillna(0)

visible_uplift_contacts = pd.DataFrame(df_part.groupby('contact_id')['treatment'].nunique())
visible_uplift_contacts = visible_uplift_contacts[visible_uplift_contacts['treatment'] == 2].reset_index()

df_part = df_part[df_part['contact_id'].isin(visible_uplift_contacts['contact_id'].values)]
df_part['target_dac'] = df_part['target_trns'] * df_part['target_login']

df_c = pd.DataFrame(df_part[df_part['treatment'] == 0].groupby('contact_id')['target_dac'].mean()).reset_index().rename(columns={'target_dac': 'target_dac_0'})
df_t = pd.DataFrame(df_part[df_part['treatment'] == 1].groupby('contact_id')['target_dac'].mean()).reset_index().rename(columns={'target_dac': 'target_dac_1'})

df_part = df_c.merge(df_t, on='contact_id')
df_part['uplift_rate'] = df_part['target_dac_1'] - df_part['target_dac_0']
df_part = df_part[['contact_id', 'uplift_rate']]

print(f'uplift_rate loaded in {time.time() - t0:.1f} sec; shape={df_part.shape}')
df = merge_block(df, df_part, 'uplift_rate')
display(df_part.head())


## 18. Fill Nulls Like `load_features`

In [ ]:
null_cols = [c for c in df.columns if any(s in c for s in ['count', 'sum', 'rto', 'aov'])]
df[null_cols] = df[null_cols].fillna(0)

print('After null fill:', df.shape)
display(df[null_cols].isna().mean().sort_values(ascending=False).head(20).to_frame('null_share'))
